<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
     <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

# Monopulse Tracking

Is a technique used in RADAR to estimate the direction of a target in a single pulse, by comparing the the received signal from **multiple** beam patterns.

In the previous example we swept the steering angle to find the direction of the target, using Monopulse tracking we can calculate the instantaneous angle error which is vital for tracking targets.

## Array Types

Right at the start we mentioned that the Phaser kit is a hybrid phased-array system.
To understand what that means, it's useful to compare the two extremes: fully-analog and fully-digital beamformers.

#### Analog Beamformer
- All beamforming happens in the analog/RF domain.
- Phase shifting, TDD switching, amplitude control and vector modulation happen before any ADC.
- The outputs of all elements (or subarrays) are combined into one RF signal before digitisation.
- Pros: low data rate, simple digital backend.
- Cons: limited flexibility, only one (or few) beams at a time.

<img src="resources/analog-bf.png" alt="Analog Beamformer" width="250">

#### Digital Beamformer

- Each antenna element is individually digitised right after the RF front-end.
- All phase shifts, delays, beamforming and algorithms happen in the digital domain (FPGA/SoC).
- Pros: maximum flexibility, multiple simultaneous beams, adaptive algorithms.
- Cons: huge data rates, high power, expensive hardware.

<img src="resources/digital-bf.png" alt="Digital Beamformer" width="250"> <br>

#### Hyrbid

A hybrid array combines the benefits of both approaches:

- Subarray-level analog beamforming (using ADAR1000s).
- Digital processing only after the subarrays have already applied analog phase/gain control.
- This significantly reduces digital data rate while still allowing adaptive techniques.

<img src="resources/hybrid-bf.png" alt="Digital Beamformer" width="500"> <br>

----

Phaser implements two analog subarrays:

- Left subarray (4 elements via one ADAR1000)
- Right subarray (4 elements via the second ADAR1000)

These two subarray outputs are then down-converted and digitised as two digital channels using the PlutoSDR.

This architecture is exactly what you want for monopulse:

## Monopulse Concepts
In a monopulse radar we need:
- A sum signal $\sum$. This is our standard beam formed from the addition of our left and right beams.
- A difference signal $\Delta$.  This is created by subracting the left and right beams

The monpulse ratio is:

$
\Large R(\theta) = \frac{\Delta(\theta)}{\sum(\theta)}
$

The sign and magnitude of $R(\theta)$  tells you:
- Which direction the target is off-center
- How far it is from the beam center

### Monopulse Methods: Magnitude and Phase Comparison

There are two principal methods for computing angle error in a monopulse radar system:

---

#### 1. **Amplitude (Magnitude) Monopulse**

This is the most common and intuitive method.

- The radar forms two sub-beams: typically a **left** and **right** beam.
- It measures the **difference in signal strength** (magnitude) between the two:

  $
  \Large R_{\text{mag}} = \frac{|L| - |R|}{|L| + |R|}
  $

- This produces a signed value:
  - **Positive** when the target is more towards the left.
  - **Negative** when it's more towards the right.
  - **Zero** when the target is centered.

- This is often linearly scaled to produce an **angle estimate**:

  $
  \Large \hat{\theta} = K \cdot R_{\text{mag}}
  $

  where \( K \) is a gain factor depending on the antenna beamwidth and geometry.

**Pros:**

- Simple to implement  
- Works well with low SNR signals

**Cons:**

- Limited angular accuracy  
- Sensitive to gain imbalance between channels

---

#### 2. **Phase Monopulse**

In systems where both left and right beams are captured as **complex signals**, you can also measure the **phase difference** between them:

- Let

  $ \Large L = |L| e^{j\phi_L}$ ,  $\Large R = |R| e^{j\phi_R}$
- Compute:

  $
  \Large R_{\text{phase}} = \arg\left( \frac{L}{R} \right) = \phi_L - \phi_R
  $

- This phase difference directly relates to angle deviation via:

  $
  \Large \hat{\theta} = \frac{\lambda}{2\pi d} \cdot R_{\text{phase}}
  $

  where:
  - $\lambda$: wavelength of the signal  
  - $d$: element spacing  
  - $R_{\text{phase}}$: measured phase difference (in radians)

**Pros:**

- High resolution (especially near boresight)  
- Immune to gain fluctuations

**Cons:**

- Phase unwrapping required for large angles  
- Requires coherent receiver (complex I/Q)  
- Sensitive to phase calibration errors

---

### Summary

| Method       | Signal Type     | Formula                             | Pros                        | Cons                              |
|--------------|------------------|-----------------------------------|-----------------------------|-----------------------------------|
| **Magnitude** | Real-only        | $ \frac{|L| - |R|}{|L| + |R|} $  | Simple, robust              | Lower angular resolution          |
| **Phase**     | Complex (I/Q)    | $ \phi_L - \phi_R $              | High resolution             | Needs calibration, unwrap ±180°   |



In [ ]:
import sys
sys.path.insert(0, '../src')

%matplotlib widget

from time import sleep
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, update_display, Markdown
from phaser_functions import set_up_phaser
from phaser_init import init_phaser_sdr
import time

plt.style.use('ggplot')
np.set_printoptions(legacy='1.25', precision=2)

# Setup the phaser hardware
try:
    myPhaser = set_up_phaser()
except Exception as e:
    print(f"Error setting up phaser: {e}")
    exit(1)

In [ ]:
def mag_mono_pulse(sig1 : np.ndarray, sig2 : np.ndarray) -> float:
    """
    Calculate the monopulse magnitude ratio Δ / Σ.
    """
    if len(sig1) != len(sig2):
        raise ValueError("Signals must be of the same length")

    mean_sig1 = np.mean(sig1)
    mean_sig2 = np.mean(sig2)

    abs_sig1 = np.abs(mean_sig1)
    abs_sig2 = np.abs(mean_sig2)

    delta = abs_sig1 - abs_sig2
    summation = abs_sig1 + abs_sig2

    if summation == 0:
        return 0.0

    return delta / summation

def mag_mono_pulse_freq(sig1: np.ndarray, sig2: np.ndarray, win: np.ndarray) -> float:
    """
    Frequency-domain magnitude monopulse Δ/Σ.
    """
    sig1_fft = np.fft.fft(sig1 * win)
    sig2_fft = np.fft.fft(sig2 * win)

    peak_bin = np.argmax(np.abs(sig1_fft))  # Or use fixed bin if tone is known

    abs1 = np.abs(sig1_fft[peak_bin])
    abs2 = np.abs(sig2_fft[peak_bin])

    summation = abs1 + abs2
    delta = abs1 - abs2

    return delta / summation if summation != 0 else 0.0

def phase_monopulse(sig1 : np.ndarray, sig2 : np.ndarray) -> float:
    """
    Calculate the monopulse phase difference in degrees.
    """
    if len(sig1) != len(sig2):
        raise ValueError("Signals must be of the same length")

    mean_sig1 = np.mean(sig1)
    mean_sig2 = np.mean(sig2)

    delta = mean_sig1 - mean_sig2
    summation = mean_sig1 + mean_sig2

    ratio = delta / summation if summation != 0 else 0.0
    phase_diff_rad = np.angle(ratio)

    return np.rad2deg(phase_diff_rad)

def phase_monopulse_freq(sig1: np.ndarray, sig2: np.ndarray, win: np.ndarray) -> float:
    """
    Calculate phase difference between sig1 and sig2 in frequency domain.
    """
    sig1_fft = np.fft.fft(sig1 * win)
    sig2_fft = np.fft.fft(sig2 * win)

    peak_bin = np.argmax(np.abs(sig1_fft))  # Use sig1 as reference
    phase_diff_rad = np.angle(sig2_fft[peak_bin]) - np.angle(sig1_fft[peak_bin])

    return np.rad2deg(np.angle(np.exp(1j * phase_diff_rad)))

def get_signal_info(sig1 : np.ndarray, sig2 : np.ndarray) -> tuple:
    """
    Get basic signal information for debugging.
    """
    left_level = np.mean(np.abs(sig1))
    right_level = np.mean(np.abs(sig2))

    left_phase_deg = np.rad2deg(np.angle(np.mean(sig1)))
    right_phase_deg = np.rad2deg(np.angle(np.mean(sig2)))

    phase_diff_deg = left_phase_deg - right_phase_deg

    return left_level, right_level, left_phase_deg, right_phase_deg, phase_diff_deg

## Investigate Magnitude and Phase Monopulse

The following code uses the SDR to read signals received from the left and right sides of the phaser array.  
We then apply signal processing techniques to examine both the **magnitude ratio** between channels and the **phase difference**.  
The results are plotted live, allowing us to observe how these metrics respond as the target's position changes.


In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(2, 2, figsize=(10, 7))
ax2_1_1 = ax[1,1].twinx()
hdisplay = display("", display_id=True)

line_mag_ratio, = ax[0,0].plot([], [], marker='o', linestyle='-')
ax[0,0].set_xlim(0, 50)
ax[0,0].set_ylim(-1.0, 1.0)
ax[0,0].set_xlabel("Acquisition Index")
ax[0,0].set_ylabel("Δ / Σ Ratio")
ax[0,0].set_title("Magnitude Δ / Σ Ratio")
ax[0,0].grid(True)

line_phase_ratio, = ax[0,1].plot([], [], marker='o', linestyle='-')
ax[0,1].set_xlim(0, 50)
ax[0,1].set_ylim(-180, 180)
ax[0,1].set_xlabel("Acquisition Index")
ax[0,1].set_ylabel("Phase Angle (deg)")
ax[0,1].set_title("Phase Difference (L - R)")
ax[0,1].grid(True)

line_signal_r, = ax[1,0].plot([], [], label='Right')
line_signal_l, = ax[1,0].plot([], [], label='Left')
ax[1,0].set_xlabel("Sample Index")
ax[1,0].set_ylabel("Signal Amplitude (Left/Right)")
ax[1,0].set_title("Read Signal (Left/Right)")
ax[1,0].grid(True)


line_signal_Δ, = ax2_1_1.plot([], [], label='Δ', color='tab:orange', linestyle='--')
line_signal_Σ, = ax[1,1].plot([], [], label='Σ')
ax[1,1].set_xlabel("Sample Index")
ax[1,1].set_ylabel("Amplitude")
ax[1,1].set_title("Δ and Σ Signals")
# ax[1,1] legend set dynamically inside update loop

K_gain = 80
K_phase = -0.2

num_samples = myPhaser.sdr.rx_buffer_size
win = np.blackman(num_samples)
win /= np.average(win)

myPhaser.set_beam_phase_diff(0)

def update_monopulse_plot(myPhaser, win, iterations=10, average_window=2):
    global mag_ratios, cplx_ratios

    mag_ratios = []
    cplx_ratios = []
    mag_window = []
    cplx_window = []
    left_mag_history = []
    right_mag_history = []
    phase_difference_history = []

    for _ in range(iterations):
        # Capture signals
        left, right = myPhaser.sdr.rx()

        # Phase-lock: rotate left to zero phase based on first sample and apply same rotation to right
        ref_phase = np.angle(left[0])
        left_fft = np.fft.fft(left * win)
        peak_bin = np.argmax(np.abs(left_fft))
        ref_phase = np.angle(left_fft[peak_bin])
        rotation = np.exp(-1j * ref_phase)
        left *= rotation
        right *= rotation
        sum_raw = left + right
        delta_raw = left - right
        #left *= win
        #right *= win

        left_level, right_level, left_phase_deg, right_phase_deg, phase_deg = get_signal_info(left, right)
        sum_raw_level, delta_raw_level, _, _, sum_delta_phase_diff_deg = get_signal_info(sum_raw, delta_raw)

        #delta_mag = mag_mono_pulse(left, right)
        delta_mag = mag_mono_pulse_freq(left, right, win)
        #phase_diff_deg = phase_monopulse(left, right)
        phase_diff_deg = phase_monopulse_freq(left, right, win)

        mag_window.append(delta_mag)
        cplx_window.append(phase_diff_deg)
        left_mag_history.append(left_level)
        right_mag_history.append(right_level)
        phase_difference_history.append(phase_deg)

        if len(mag_window) > average_window:
            mag_window.pop(0)
        if len(cplx_window) > average_window:
            cplx_window.pop(0)

        mag_avg = np.mean(mag_window) if mag_window else 0.0
        phase_avg = np.mean(cplx_window) if cplx_window else 0.0

        mag_ratios.append(mag_avg)
        cplx_ratios.append(phase_avg)

        doa_mag = mag_avg * K_gain
        doa_phase = phase_avg * K_phase

        window_size = 50
        line_mag_ratio.set_data(range(len(mag_ratios[-window_size:])), mag_ratios[-window_size:])
        line_phase_ratio.set_data(range(len(cplx_ratios[-window_size:])), cplx_ratios[-window_size:])
        line_signal_l.set_data(range(len(left)), left.real)
        line_signal_r.set_data(range(len(right)), right.real)
        line_signal_Δ.set_data(range(len(left)), (delta_raw).real)
        line_signal_Σ.set_data(range(len(left)), (sum_raw).real)
                
        ax[0,0].set_xlim(0, window_size)
        ax[0,1].set_xlim(0, window_size)
        ax[1,0].set_xlim(0, len(left))
        ax[1,0].set_ylim(-1500, 1500)
        ax[1,1].set_xlim(0, len(left))
        ax[1,1].set_ylim(-2000, 2000)
        ax2_1_1.set_ylim(-2000, 2000)

        
        fig.tight_layout()

        # Update legend in [1,0] with live signal info
        line_signal_l.set_label(f"Left (lvl={left_level:.1f})")
        line_signal_r.set_label(f"Right (lvl={right_level:.1f})")
        custom_phase = Line2D([], [], linestyle='None', label=f"Phase diff = {phase_avg:.1f}°")
        ax[1,0].legend(handles=[line_signal_l, line_signal_r, custom_phase], loc='upper right')

        line_signal_Δ.set_label(f"Δ (lvl={delta_raw_level:.2f})")
        line_signal_Σ.set_label(f"Σ (lvl={sum_raw_level:.2f})")
        ax[1,1].legend(handles=[
            line_signal_Σ,
            line_signal_Δ
        ], loc='upper right')

        info_md = f"""
### 💉 Monopulse Live Diagnostics
**Iteration:** {len(mag_ratios)}  
**Beam Angle (set):** 0° (boresight)  
**Estimated DOA (Mag):** {doa_mag:+.2f}°  
**Estimated DOA (Phase):** {doa_phase:+.2f}°  
**Phase Δ/Σ (deg):** {phase_deg:+.2f}  
**Left Amp:** {left_level:.2f} | **Right Amp:** {right_level:.2f}  
**Δ/Σ Ratio:** {mag_avg:.4f}  
"""
        hdisplay.update(fig)
        update_display(Markdown(info_md), display_id="status")
        time.sleep(0.1)



display(Markdown("### Monopulse Live Diagnostics\nInitialising..."), display_id="status")
update_monopulse_plot(myPhaser, win, iterations=2000);


## Let's Close the Loop

In this section, we implement a closed-loop monopulse tracker.

We use the **magnitude monopulse technique** to estimate the target’s position and update the beam direction accordingly. The loop continuously adjusts the beam until the estimated direction of arrival (DOA) error is minimized.

The goal is to achieve **target error ≈ 0°** by dynamically steering the beam toward the signal source.


In [ ]:
# Close loop Monopulse
Kp = 100  # Negative gain: positive Δ/Σ (target left) produces negative correction (steer left)
Ki = 0.1
integral_error = 0.0

beam_angle = 0.0  # start at boresight
doa_history = []
beam_history = []
delta_history = []

average_window = 1  # moving average window

# Setup plot
fig, ax = plt.subplots(figsize=(10, 5))
line1, = ax.plot([], [], label="Beam Correction (deg)")
line2, = ax.plot([], [], label="Beam Angle (deg)")

ax.set_xlim(0, 100)
ax.set_ylim(-80, 80)
ax.set_xlabel("Iteration")
ax.set_ylabel("Angle (degrees)")
ax.set_title("Monopulse DOA Tracking (Magnitude-Based)")
ax.grid(True)

# Add second y-axis for delta ratio
ax2 = ax.twinx()
line3, = ax2.plot([], [], 'r--', label="Δ/Σ Ratio")
ax2.set_ylabel("Δ/Σ Ratio")
ax2.set_ylim(-0.25, 0.25)

# Combine legends
lines = [line1, line2, line3]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc="upper right")

fig.tight_layout()

hplot = display(fig, display_id=True)
hstatus = display(Markdown("### Monopulse DOA Tracker Initialising..."), display_id="status")

num_samples = myPhaser.sdr.rx_buffer_size
win = np.blackman(num_samples)
win /= np.average(win)

# Start tracking loop
iteration = 0
delta_window = []

try:
    while True:

        myPhaser.set_beam_phase_diff(beam_angle)
        time.sleep(0.2)  # Allow hardware to settle before reading
        
        # Receive phaser signal
        left, right = myPhaser.sdr.rx()

        # left_level, right_level, left_phase_deg, right_phase_deg, phase_deg = get_signal_info(left, right)

        delta_ratio = mag_mono_pulse_freq(left, right, win)
        #delta_ratio = phase_monopulse_freq(left, right, win)

        # Apply simple moving average
        delta_window.append(delta_ratio)
        if len(delta_window) > average_window:
            delta_window.pop(0)
        delta_avg = np.mean(delta_window)

        # Calculate beam correction (proportional control signal)
        beam_correction = delta_avg * Kp

        integral_error += delta_avg
        integral_error = np.clip(integral_error, -10, 10)

        beam_angle += beam_correction + Ki * integral_error
        beam_angle = np.clip(beam_angle, -70, 70)

        # Store history
        doa_history.append(beam_correction)
        beam_history.append(beam_angle)
        delta_history.append(delta_avg)

        # Update plot
  
        window_size = 50
        start_idx = max(0, iteration - window_size)
        x = range(start_idx, iteration)

        line1.set_data(x, doa_history[start_idx:iteration])
        line2.set_data(x, beam_history[start_idx:iteration])
        line3.set_data(x, delta_history[start_idx:iteration])

        if iteration > 0:
            ax.set_xlim(start_idx, iteration)
        else:
            ax.set_xlim(0, 1)
            
        ax.set_ylim(-80, 80)
        ax2.set_ylim(-1.0, 1.0)

        hplot.update(fig)

        # Live diagnostics
        info = f"""
### 🛱 Monopulse DOA Live Diagnostics
**Iteration:** {iteration}  
**Δ/Σ Ratio (avg):** `{delta_avg:+.4f}`  
**Beam Correction:** `{beam_correction:+.2f}°`  
**Updated Beam Angle:** `{beam_angle:+.2f}°`
"""
        update_display(Markdown(info), display_id="status")

        iteration += 1
        #time.sleep(0.2)

except KeyboardInterrupt:
    myPhaser.set_beam_phase_diff(0)
    print("Tracking stopped by user.")

plt.close(fig)

##  Summary: Monopulse DOA Tracking and Closed-Loop Beam Steering

In this notebook, we've explored real-time Direction of Arrival (DOA) tracking using the **monopulse technique**. The key steps were:

-  **Monopulse Measurement**: Calculated the Δ/Σ ratio from received signals to estimate angular deviation from boresight.
-  **Closed-Loop Control**: Applied a PI controller to update the beam angle in response to the estimated DOA error.
-  **Target Locking**: Demonstrated automatic beam steering that reduces angular error and locks onto a moving source.
-  **Live Diagnostics**: Visualized DOA estimates, beam angle adjustments, and Δ/Σ ratios in real-time.

This method enables fast and responsive tracking of signal sources, with minimal computational overhead and good robustness near boresight. The addition of integral control further reduces steady-state tracking errors.

